In [1]:
# %% 1. 建立 KITTI 的 YOLO 設定檔與轉換
import os
import yaml
import cv2
import glob

dataset_path = './kitti_dataset' # 替換為實際路徑
kitti_yaml = {
    'path': dataset_path,
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'names': {
        0: 'Car', 1: 'Van', 2: 'Truck', 3: 'Pedestrian',
        4: 'Person_sitting', 5: 'Cyclist', 6: 'Tram', 7: 'Misc', 8: 'DontCare'
    }
}

os.makedirs('dataset_configs', exist_ok=True)
with open('dataset_configs/kitti.yaml', 'w') as f:
    yaml.dump(kitti_yaml, f, default_flow_style=False)
print("KITTI 設定檔已建立於 dataset_configs/kitti.yaml")

KITTI 設定檔已建立於 dataset_configs/kitti.yaml


In [ ]:
def convert_kitti_to_yolo(kitti_label_path, image_path, yolo_label_path, name_to_id):
    os.makedirs(os.path.dirname(yolo_label_path), exist_ok=True)
    img = cv2.imread(image_path)
    if img is None:
        return
    img_h, img_w = img.shape[:2]

    with open(kitti_label_path, 'r') as f_in, open(yolo_label_path, 'w') as f_out:
        for line in f_in:
            parts = line.strip().split(' ')
            cls_name = parts[0]
            if cls_name not in name_to_id:
                continue
            
            cls_id = name_to_id[cls_name]
            xmin, ymin, xmax, ymax = map(float, parts[4:8])
            
            x_center = ((xmin + xmax) / 2) / img_w
            y_center = ((ymin + ymax) / 2) / img_h
            width = (xmax - xmin) / img_w
            height = (ymax - ymin) / img_h
            
            f_out.write(f"{cls_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}\n")

# 建立反向字典用於轉換 (Name -> ID)
name_to_id_map = {v: k for k, v in kitti_yaml['names'].items()}

splits = ['train', 'val']
for split in splits:
    img_dir = os.path.join(dataset_path, f'images/{split}')
    label_dir = os.path.join(dataset_path, f'kitti_labels/{split}') 
    yolo_label_dir = os.path.join(dataset_path, f'labels/{split}')
    
    if os.path.exists(label_dir):
        label_files = glob.glob(os.path.join(label_dir, '*.txt'))
        for label_file in label_files:
            base_name = os.path.basename(label_file).replace('.txt', '.png')
            img_file = os.path.join(img_dir, base_name)
            yolo_file = os.path.join(yolo_label_dir, os.path.basename(label_file))
            convert_kitti_to_yolo(label_file, img_file, yolo_file, name_to_id_map)


In [3]:
# %% 2. 模型訓練 (使用 YOLO26 特性)
from ultralytics import YOLO

# 載入 YOLO26 模型
model = YOLO('yolo26n.pt') 

epochs = 100
batch_size = 32
img_size = 640

print("開始在 KITTI 資料集上訓練模型...")
results = model.train(
    data='dataset_configs/kitti.yaml',
    epochs=epochs,
    imgsz=img_size,
    batch=batch_size,
    device=0, 
    optimizer='MuSGD', # 採用 YOLO26 的 MuSGD 優化器以提高收斂穩定度
    lr0=0.001
)
print("訓練完成。")


開始在 KITTI 資料集上訓練模型...
Ultralytics 8.4.60 🚀 Python-3.10.12 torch-2.12.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset_configs/kitti.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-8, nbs=64, nms=False, opset=None, optimize=False, opt

In [9]:
best_model_path = 'runs/detect/train/weights/best.pt'
model = YOLO(best_model_path)

print("評估 PyTorch 原始模型在驗證集上的表現 (mAP)...")
val_results = model.val(data='dataset_configs/kitti.yaml', split='val', device=0)
print(f"mAP50-95: {val_results.box.map:.4f}")
print(f"mAP50: {val_results.box.map50:.4f}")

print("將模型匯出為 TensorRT (FP16)...")
# YOLO26 為端到端架構，明確設置 nms=False 避免插入多餘後處理節點
model.export(format='engine', half=True, device=0, nms=False) 

trt_model_path = 'runs/detect/train/weights/best.engine'
optimized_model = YOLO(trt_model_path)
print("加速模型載入完成。")

評估 PyTorch 原始模型在驗證集上的表現 (mAP)...
Ultralytics 8.4.60 🚀 Python-3.10.12 torch-2.12.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
YOLO26n summary (fused): 122 layers, 2,376,591 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access ✅ (ping: 4.3±1.0 ms, read: 8.0±1.2 MB/s, size: 48.4 KB)
val: Scanning /mnt/c/Users/hodso/OneDrive/桌面/作業/人工智慧/final_project/datasets/kitti_dataset/labels/val.cache... 1496 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1496/1496 261.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 94/94 18.2it/s 5.2s0.1s
                   all       1496       8128      0.849      0.712      0.816      0.572
                   Car       1322       5716      0.875      0.879      0.939      0.739
                   Van        419        555      0.852      0.806      0.892       0.67
                 Truck        212        223      0.902      0.861      0.933      0.748
            Pede

/home/hodso/.local/lib/python3.10/site-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset11.py:954: UserWarning: Exporting aten::index operator of advanced indexing in opset 18 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  return opset9.index(g, self, index)


ONNX: slimming with onnxslim 0.1.94...
ONNX: export success ✅ 0.9s, saved as 'runs/detect/train/weights/best.onnx' (9.4 MB)

TensorRT: starting export with TensorRT 10.15.1.29...
[06/04/2026-02:24:53] [TRT] [W] WARNING The logger passed into createInferBuilder differs from one already registered for an existing builder, runtime, or refitter. So the current new logger is ignored, and TensorRT will use the existing one which is returned by nvinfer1::getLogger() instead.
[06/04/2026-02:24:53] [TRT] [I] ----------------------------------------------------------------
[06/04/2026-02:24:53] [TRT] [I] Input filename:   runs/detect/train/weights/best.onnx
[06/04/2026-02:24:53] [TRT] [I] ONNX IR version:  0.0.8
[06/04/2026-02:24:53] [TRT] [I] Opset version:    18
[06/04/2026-02:24:53] [TRT] [I] Producer name:    pytorch
[06/04/2026-02:24:53] [TRT] [I] Producer version: 2.12.0
[06/04/2026-02:24:53] [TRT] [I] Domain:           
[06/04/2026-02:24:53] [TRT] [I] Model version:    0
[06/04/2026-02:24

In [10]:
import time
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dummy_frame = torch.randn(1, 3, 640, 640).to(device)

print("預熱模型中 (Warm-up)...")
for _ in range(50): # 增加預熱次數以確保 CUDA 穩定
    _ = optimized_model(dummy_frame, verbose=False)

print("開始效能測試...")
# 使用 CUDA event 來獲得更精確的 GPU 計時
start_event = torch.cuda.Event(enable_timing=True)
end_event = torch.cuda.Event(enable_timing=True)

num_frames = 500

start_event.record()
for _ in range(num_frames):
    _ = optimized_model(dummy_frame, verbose=False)
end_event.record()

torch.cuda.synchronize() # 等待 GPU 運算完成
total_time_ms = start_event.elapsed_time(end_event)

total_time_s = total_time_ms / 1000.0
fps = num_frames / total_time_s
latency = total_time_ms / num_frames

print(f"評估結果:")
print(f"平均推論延遲 (Latency): {latency:.2f} ms")
print(f"每秒幀數 (FPS): {fps:.2f}")

if fps >= 10:
    print("符合提案中最低 10 FPS 之要求。")
else:
    print("未達最低 10 FPS 門檻。")

預熱模型中 (Warm-up)...
Loading runs/detect/train/weights/best.engine for TensorRT inference...
[06/04/2026-02:28:12] [TRT] [I] Loaded engine size: 7 MiB
[06/04/2026-02:28:12] [TRT] [I] [MemUsageChange] TensorRT-managed allocation in IExecutionContext creation: CPU +0, GPU +9, now: CPU 0, GPU 13 (MiB)
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 5.565520763397217. Dividing input by 255.
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 5.565520763397217. Dividing input by 255.
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 5.565520763397217. Dividing input by 255.
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 5.565520763397217. Dividing input by 255.
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 5.565520763397217. Dividing input by 255.
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 5.565520763397217. Dividing in

In [2]:
from ultralytics import YOLO

# 1. 載入剛剛加速完成的 TensorRT 模型
model = YOLO('runs/detect/train/weights/best.engine')

# 2. 設定您的影片路徑 (請替換為實際的影片檔案名稱)
video_path = 'road.mp4' 

# 3. 執行預測
results = model.predict(
    source=video_path,
    conf=0.3,           # 信心分數閾值 (數值大於 0.4 的預測框才會顯示)
    show=True,          # 執行時同步彈出視窗即時顯示畫面
    save=True,          # 儲存標註後的完整影片
    device=0            # 使用 GPU
)

print("影片推論完成！標註後的影片已儲存至 runs/detect/predict 目錄下。")

QFontDatabase: Cannot find font directory /home/hodso/.local/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/hodso/.local/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/hodso/.local/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/hodso/.local/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/hodso/.local/lib/python3.10/site-package

Loading runs/detect/train/weights/best.engine for TensorRT inference...
[06/04/2026-20:56:39] [TRT] [I] Loaded engine size: 7 MiB
[06/04/2026-20:56:39] [TRT] [I] [MemUsageChange] TensorRT-managed allocation in IExecutionContext creation: CPU +0, GPU +9, now: CPU 0, GPU 13 (MiB)

WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/9030) /mnt/c/Users/hodso/OneDrive/桌面/作業/人工智慧/final_project/road.mp4: 640x640 1 Car, 1.6ms
video 1/1 (frame 2/9030) /mnt/c/Users/hodso/OneDrive/桌面/作業/人工智慧/final_

KeyboardInterrupt: 